# 05 — Sensitivity Analysis

**Study:** Understanding the Structure and Reuse of AI Agent Skills in Open-Source Software Projects: An Empirical Study  
**Dataset:** GitSkills sample (July 2026)  
**Research question:** RQ2 — To what extent are identical AI agent skills reused across different repositories?

This notebook examines whether the cross-repository overlap reported in the paper changes when the analyzed repositories or files are restricted. We consider repositories with known fork metadata, remove the largest repositories by skill-file count, and restrict the analysis to files named exactly `SKILL.md`.

These comparisons help assess the stability of the descriptive results. They do not establish how a skill reached a repository or whether its appearance represents intentional reuse. The analysis uses the local SQLite sample and can be run independently of the other notebooks.


## 1. Data access

The analysis reads `data/raw/agent_skills_sample.db` from the project directory. The database is opened in read-only mode. The required Python packages are pandas and NumPy; SQLite is included in Python's standard library.

The notebook can be run from the project root or its `notebooks/` directory. No files are modified during the analysis.


In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

db_path = project_root / "data" / "raw" / "agent_skills_sample.db"
if not db_path.is_file():
    raise FileNotFoundError(f"Database not found: {db_path}")

conn = sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)
print("Database:", db_path)
print("SQLite version:", sqlite3.sqlite_version)

Database: C:\Users\mouzh\Documents\GitHub\MSR2027-GitSkills\data\raw\agent_skills_sample.db
SQLite version: 3.49.1


## 2. Baseline overlap

We measure overlap using the Git blob hash stored in `file_sha`. Each row in `artifacts` represents a recorded file occurrence. A content hash is considered shared across repositories only if it appears under at least two distinct `repo_full_name` values; multiple copies within a single repository do not qualify.

Two proportions are reported. **Hash-level overlap** is the share of distinct hashes found in more than one repository. **Occurrence-level overlap** is the share of all artifact rows belonging to those shared hashes. The baseline includes all recorded occurrences and provides the reference for the sensitivity checks.


In [2]:
artifact_hashes = pd.read_sql_query(
    "SELECT repo_full_name, file_sha, filename FROM artifacts", conn
)
if artifact_hashes[["repo_full_name", "file_sha"]].isna().any().any():
    raise ValueError("Null repository or file_sha values would invalidate overlap grouping")

def calculate_overlap(frame):
    """Compute exact-content overlap using repository-distinct hash groups."""
    if frame.empty:
        raise ValueError("Cannot calculate overlap on an empty population")
    groups = frame.groupby("file_sha").agg(
        occurrences=("repo_full_name", "size"),
        repositories=("repo_full_name", "nunique"),
    )
    shared = groups["repositories"] >= 2
    shared_hashes = int(shared.sum())
    shared_occurrences = int(groups.loc[shared, "occurrences"].sum())
    return {
        "Repositories": int(frame["repo_full_name"].nunique()),
        "Artifacts": int(len(frame)),
        "Unique hashes": int(len(groups)),
        "Shared hashes": shared_hashes,
        "Shared hashes (%)": 100 * shared_hashes / len(groups),
        "Artifacts with shared content": shared_occurrences,
        "Artifact overlap (%)": 100 * shared_occurrences / len(frame),
    }

baseline = calculate_overlap(artifact_hashes)
display(pd.DataFrame({"Baseline": baseline}).round(2))

assert baseline["Artifacts"] == 29_786
assert baseline["Unique hashes"] == 13_000
assert baseline["Repositories"] == 11_841
assert baseline["Shared hashes"] == 2_854
assert baseline["Artifacts with shared content"] == 18_855

,Baseline
Repositories,11841.00
Artifacts,29786.00
Unique hashes,13000.00
Shared hashes,2854.00
Shared hashes (%),21.95
Artifacts with shared content,18855.00
Artifact overlap (%),63.30


## 3. Repositories with known fork status

The `repos` table includes an `is_fork` field, but its values are not complete for every repository. We first inspect the recorded statuses, then recalculate overlap for repositories explicitly marked as non-forks (`is_fork = 0`).

No repository in this sample is explicitly marked as a fork. Consequently, the restricted analysis mainly removes repositories with missing fork information. It should not be interpreted as a comparison between forked and independently developed repositories.


In [3]:
fork_metadata = pd.read_sql_query(
    "SELECT is_fork, COUNT(*) AS repositories FROM repos GROUP BY is_fork ORDER BY is_fork",
    conn,
)
artifact_fork = pd.read_sql_query(
    """SELECT r.is_fork, COUNT(*) AS artifacts,
              COUNT(DISTINCT a.repo_full_name) AS repositories
       FROM artifacts AS a LEFT JOIN repos AS r
       ON a.repo_full_name = r.full_name
       GROUP BY r.is_fork ORDER BY r.is_fork""",
    conn,
)
print("Repository-level fork metadata:")
display(fork_metadata)
print("Artifact-level fork metadata:")
display(artifact_fork)

Repository-level fork metadata:


,is_fork,repositories
0,NaN,37
1,0.0,11804


Artifact-level fork metadata:


,is_fork,artifacts,repositories
0,NaN,67,37
1,0.0,29719,11804


In [4]:
nonfork_artifacts = pd.read_sql_query(
    """SELECT a.repo_full_name, a.file_sha
       FROM artifacts AS a
       INNER JOIN repos AS r ON a.repo_full_name = r.full_name
       WHERE r.is_fork = 0""",
    conn,
)
nonfork = calculate_overlap(nonfork_artifacts)
display(pd.DataFrame({"All repositories": baseline, "Explicit non-forks": nonfork}).round(2))
print("Artifact-overlap change (percentage points):",
      f"{nonfork['Artifact overlap (%)'] - baseline['Artifact overlap (%)']:+.2f}")
print("Interpretation: exclusion of unknown-status records, not removal of known forks.")

,All repositories,Explicit non-forks
Repositories,11841.00,11804.00
Artifacts,29786.00,29719.00
Unique hashes,13000.00,12997.00
Shared hashes,2854.00,2851.00
Shared hashes (%),21.95,21.94
Artifacts with shared content,18855.00,18787.00
Artifact overlap (%),63.30,63.22


Artifact-overlap change (percentage points): -0.09
Interpretation: exclusion of unknown-status records, not removal of known forks.


### Result: fork-status restriction

The occurrence-level overlap is **63.22%** among repositories explicitly marked as non-forks, compared with **63.30%** in the full sample. The small difference shows that excluding records with unknown fork status has little effect on this estimate. It does not rule out repository mirroring or other relationships that are not captured by the metadata.


## 4. Excluding the largest repositories

Repository sizes vary considerably in the number of recorded skill files. To examine the influence of large collections, we rank repositories by artifact count and remove the largest 1%, rounding the number of excluded repositories upward. The overlap measures are then recalculated using only the remaining occurrences.

This changes the analyzed population deliberately; it is not a random subsample. The test measures sensitivity to repository-size concentration, not whether particular repositories are mirrors or templates.


In [5]:
repo_sizes = (
    artifact_hashes.groupby("repo_full_name", as_index=False)
    .size().rename(columns={"size": "artifact_count"})
    .sort_values(["artifact_count", "repo_full_name"], ascending=[False, True])
)
top_n = max(1, int(np.ceil(len(repo_sizes) * 0.01)))
excluded_repos = set(repo_sizes.head(top_n)["repo_full_name"])
excluded_count = int(repo_sizes.head(top_n)["artifact_count"].sum())

print("Total repositories:", len(repo_sizes))
print("Excluded repositories (top 1%, rounded up):", top_n)
print("Excluded artifact occurrences:", excluded_count)
print("Share of artifact occurrences excluded:", f"{100 * excluded_count / len(artifact_hashes):.2f}%")
print("Largest repositories (inspection only):")
display(repo_sizes.head(10))

Total repositories: 11841
Excluded repositories (top 1%, rounded up): 119
Excluded artifact occurrences: 12606
Share of artifact occurrences excluded: 42.32%
Largest repositories (inspection only):


,repo_full_name,artifact_count
8127,majiayu000/claude-skill-registry-data,2952
5537,diegosouzapw/awesome-omni-skill,1886
2249,MikeCheng1208/BattleTree,813
927,David-Li0406/meta-skill-evloving,728
8126,majiayu000/claude-skill-registry,593
2415,NeuralBlitz/ncx,422
6292,gabrielmoreira/agent-skills-mirror,416
2414,NeuralBlitz/buggy,204
10880,tools-only/X-Skills,164
5711,dvcrn/openclaw-skills-marketplace,155


In [6]:
filtered_artifacts = artifact_hashes.loc[
    ~artifact_hashes["repo_full_name"].isin(excluded_repos),
    ["repo_full_name", "file_sha"],
].copy()
size_filtered = calculate_overlap(filtered_artifacts)
display(pd.DataFrame({"All repositories": baseline,
                      "Excluding top 1%": size_filtered}).round(2))
size_change = round(size_filtered["Artifact overlap (%)"], 2) - round(baseline["Artifact overlap (%)"], 2)
print(f"Change in artifact-level overlap: {size_change:+.2f} percentage points")

assert top_n == 119
assert excluded_count == 12_606
assert size_filtered["Artifacts"] == 17_180
assert size_filtered["Unique hashes"] == 9_979
assert size_filtered["Shared hashes"] == 1_131
assert size_filtered["Artifacts with shared content"] == 7_775

,All repositories,Excluding top 1%
Repositories,11841.00,11722.00
Artifacts,29786.00,17180.00
Unique hashes,13000.00,9979.00
Shared hashes,2854.00,1131.00
Shared hashes (%),21.95,11.33
Artifacts with shared content,18855.00,7775.00
Artifact overlap (%),63.30,45.26


Change in artifact-level overlap: -18.04 percentage points


### Result: repository-size restriction

The 119 largest repositories account for **42.32%** of the recorded artifact occurrences. After their removal, occurrence-level overlap falls from **63.30%** to **45.26%**, a difference of **18.04 percentage points when calculated from the displayed two-decimal percentages**.

Large repositories therefore contribute substantially to the baseline estimate. Exact-content overlap remains common in the restricted sample, although neither estimate should be generalized to a random population of GitHub repositories: the GitSkills sample was selected by content hash rather than by repository.


## 5. Restricting filenames to `SKILL.md`

The original collection includes a small number of filename variants. We repeat the overlap calculation using only occurrences whose filename matches `SKILL.md` exactly, with case-sensitive matching.

This check tests whether those filename variants affect the reported overlap. It does not determine whether each file complies with the Agent Skills specification.


In [7]:
filename_distribution = pd.read_sql_query(
    """SELECT filename, COUNT(*) AS artifacts,
              COUNT(DISTINCT file_sha) AS unique_hashes,
              COUNT(DISTINCT repo_full_name) AS repositories
       FROM artifacts GROUP BY filename ORDER BY artifacts DESC""",
    conn,
)
display(filename_distribution)

exact_skill_artifacts = artifact_hashes.loc[
    artifact_hashes["filename"] == "SKILL.md",
    ["repo_full_name", "file_sha"],
].copy()
exact_skill = calculate_overlap(exact_skill_artifacts)
display(pd.DataFrame({"All filenames": baseline,
                      "Exact SKILL.md": exact_skill}).round(2))
print("Artifact-overlap change (percentage points):",
      f"{exact_skill['Artifact overlap (%)'] - baseline['Artifact overlap (%)']:+.2f}")

assert exact_skill["Artifacts"] == 29_408
assert exact_skill["Unique hashes"] == 12_785
assert exact_skill["Repositories"] == 11_726
assert exact_skill["Shared hashes"] == 2_817
assert exact_skill["Artifacts with shared content"] == 18_647

,filename,artifacts,unique_hashes,repositories
0,SKILL.md,29408,12785,11726
1,skill.md,349,309,123
2,Skill.md,24,22,9
3,SKILL.MD,5,5,2


,All filenames,Exact SKILL.md
Repositories,11841.00,11726.00
Artifacts,29786.00,29408.00
Unique hashes,13000.00,12785.00
Shared hashes,2854.00,2817.00
Shared hashes (%),21.95,22.03
Artifacts with shared content,18855.00,18647.00
Artifact overlap (%),63.30,63.41


Artifact-overlap change (percentage points): +0.11


### Result: exact filename restriction

The exact-name subset contains **29,408 occurrences** representing **12,785 distinct hashes**. Its occurrence-level overlap is **63.41%**, close to the **63.30%** baseline; hash-level overlap is **22.03%**. The result suggests that filename variants have little influence on the overlap estimates in this sample.


## 6. Comparison of sensitivity results

The table below brings the baseline and three restricted populations together. Each overlap percentage uses the number of hashes or occurrences in its own subset as the denominator. Percentage-point changes are calculated relative to the baseline occurrence-level overlap, using the displayed two-decimal percentages.


In [8]:
rows = {
    "Baseline (all artifacts)": baseline,
    "Explicit non-forks only": nonfork,
    "Exclude largest 1% of repositories": size_filtered,
    "Exact SKILL.md only": exact_skill,
}
summary = pd.DataFrame.from_dict(rows, orient="index")
summary["Artifact overlap change (pp)"] = (
    summary["Artifact overlap (%)"].round(2) - round(baseline["Artifact overlap (%)"], 2)
)
display(summary.round(2))

,Repositories,Artifacts,Unique hashes,Shared hashes,Shared hashes (%),Artifacts with shared content,Artifact overlap (%),Artifact overlap change (pp)
Baseline (all artifacts),11841,29786,13000,2854,21.95,18855,63.30,0.00
Explicit non-forks only,11804,29719,12997,2851,21.94,18787,63.22,-0.08
Exclude largest 1% of repositories,11722,17180,9979,1131,11.33,7775,45.26,-18.04
Exact SKILL.md only,11726,29408,12785,2817,22.03,18647,63.41,0.11


## 7. Limitations and reproducibility

The overlap measure identifies identical Git blob contents, not semantic similarity or the direction of copying. The GitSkills sample consists of selected content hashes and their recorded occurrences, rather than a random sample of repositories. This limits generalization from repository-level proportions.

The fork-status analysis is constrained by missing metadata and the absence of explicitly identified forks. Excluding large repositories and restricting filenames both change the observed population; they are sensitivity checks rather than alternative estimates of a broader population.

The assertions in this notebook check agreement with the counts reported for the July 2026 sample. If the underlying database changes, the assertions may need to be revisited. The original dataset and its provenance should be cited alongside any results derived from this notebook.


In [9]:
conn.close()
print("Read-only SQLite connection closed. Sensitivity analysis complete.")

Read-only SQLite connection closed. Sensitivity analysis complete.
